In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ tanh(x) = t(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}} = $$
$$ \frac{e^x}{e^x} \cdot \frac{e^x - e^{-x}}{e^x + e^{-x}} = $$
$$ \frac{e^{2x} - 1}{e^{2x} + 1} = $$
$$ \frac{2e^{2x}}{e^{2x} + 1} - \frac{e^{2x} + 1}{e^{2x} + 1} = $$
$$ 2 \cdot \text{sigmoid}(2x) - 1 $$
$$ \frac{\partial t}{\partial x} = 1 - t^2 $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from common import assert_eq, T # type: ignore
import sigmoid # type: ignore

def tanh(x):
    x = T(x)
    return 2 * sigmoid.sigmoid(2 * x) - 1
    

class TanhFunction(autograd.Function):
    @staticmethod
    def forward(ctx, x: tr.Tensor) -> tr.Tensor:
        t = tanh(x)  
        ctx.save_for_backward(t)
        return t

    @staticmethod
    def backward(ctx, dF_df: tr.Tensor) -> tuple[tr.Tensor, ]:
        (t, ) = ctx.saved_tensors

        df_dx = 1 - t ** 2
        dF_dx = dF_df * df_dx

        return (dF_dx, )
    

class Tanh(nn.Module):
    def __init__(self) -> None:
        super().__init__()

    def forward(self, x: tr.Tensor) -> tr.Tensor:
        return TanhFunction.apply(x)


def test_tanh_1() -> None:
    assert_eq(tanh(1000), 1.0)
    assert_eq(tanh(-1000), -1.0)
    assert_eq(tanh(0), 0.0)
    assert_eq(tanh(1), 0.762, 0.001)
    assert_eq(tanh(-1), -0.762, 0.001)


def test_tanh_2() -> None:
    tr.manual_seed(0)

    samples = 10
    x = tr.randn(samples, dtype=tr.float32, requires_grad=True)

    x1 = x.clone().detach().requires_grad_(True)
    y1 = TanhFunction.apply(x1)
    F1 = y1.sum()
    F1.backward()

    x2 = x.clone().detach().requires_grad_(True)
    y2 = tr.tanh(x2)
    F2 = y2.sum()
    F2.backward()

    assert_eq(y1, y2, atol=0.001)
    assert_eq(x1.grad, x2.grad, atol=0.001)


if __name__ == "__main__":
    test_tanh_1()
    test_tanh_2()
